<a href="https://colab.research.google.com/github/EMADUDDINAsdaq/federated-learning-fairness-xray/blob/main/fedavg_notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Federated Learning — Method 1: FedAvg (McMahan et al. 2017)
 Emaduddin Asdaq Syed Mohammed | CSC8639 MSc Data Science and AI  

---
**This notebook runs FedAvg only on the full dataset.**  


## Section 1 — Environment Setup

In [ ]:
pip install flwr protobuf

In [ ]:
!pip install "flwr[simulation]" protobuf -q
print("✓ Libraries installed")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 MB 35.1 MB/s eta 0:00:00
✓ Libraries installed


In [ ]:
import flwr as fl
print(f"flwr : {fl.__version__}")
print("✓ Flower working")

flwr : 1.32.1
✓ Flower working


In [ ]:
import os, json, time, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
from sklearn.metrics import roc_auc_score
import flwr as fl
from flwr.common import (NDArrays, Scalar, Parameters,
                          parameters_to_ndarrays, ndarrays_to_parameters,
                          FitIns, FitRes, EvaluateIns, EvaluateRes)
from flwr.server.strategy import Strategy
from flwr.server.client_proxy import ClientProxy
from typing import Dict, List, Optional, Tuple
warnings.filterwarnings('ignore')

print(f"flwr  : {fl.__version__}")
print(f"torch : {torch.__version__}")
print(f"numpy : {np.__version__}")
print(f"GPU   : {torch.cuda.is_available()}")

flwr  : 1.32.1
torch : 2.11.0+cu128
numpy : 2.0.2
GPU   : True


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import flwr.simulation
print("Simulation module loaded")

Simulation module loaded


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("=== Session Initialisation ===")
print(f"Device : {device}")
if torch.cuda.is_available():
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
    print(f"Memory : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
    print(f"CUDA   : {torch.version.cuda}")
    print("\n✓ GPU ready")
else:
    print("\n⚠ No GPU — Runtime → Change runtime type → A100")

=== Session Initialisation ===
Device : cuda
GPU    : NVIDIA L4
Memory : 23.7 GB
CUDA   : 12.8

✓ GPU ready


## Section 2 — Dataset Download and Image Indexing

In [ ]:
import shutil

os.makedirs('/root/.kaggle', exist_ok=True)
shutil.copy('/content/drive/MyDrive/dissertation/kaggle.json',
            '/root/.kaggle/kaggle.json')
os.chmod('/root/.kaggle/kaggle.json', 0o600)
print("✓ Kaggle credentials loaded")

os.system('pip install -q kaggle')
os.system('kaggle datasets download -d nih-chest-xrays/data '
          '--path /content/nih_kaggle --unzip --quiet')

DATASET_PATH = '/content/nih_kaggle'
print(f"✓ Dataset path: {DATASET_PATH}")

✓ Kaggle credentials loaded
✓ Dataset path: /content/nih_kaggle


## Section 3 — Load Frozen Splits

In [ ]:
SPLIT_DIR = '/content/drive/MyDrive/dissertation/splits'
HOSPITAL_NAMES = ['Hospital_A', 'Hospital_B', 'Hospital_C',
                  'Hospital_D', 'Hospital_E']
NUM_CLIENTS = 5

train_clients = {n: pd.read_csv(f'{SPLIT_DIR}/{n}_train.csv') for n in HOSPITAL_NAMES}
val_clients   = {n: pd.read_csv(f'{SPLIT_DIR}/{n}_val.csv')   for n in HOSPITAL_NAMES}
test_clients  = {n: pd.read_csv(f'{SPLIT_DIR}/{n}_test.csv')  for n in HOSPITAL_NAMES}
#Hospital_A_test.csv
for name in HOSPITAL_NAMES:
    print(f"{name}: {len(train_clients[name]):,} train / "
          f"{len(val_clients[name]):,} val / {len(test_clients[name]):,} test")

sample_path = train_clients['Hospital_A']['image_path'].iloc[0]
assert os.path.exists(sample_path), f"Path not found: {sample_path} — check Kaggle download completed"
print("✓ Image paths resolve correctly in this session")

Hospital_A: 52,332 train / 6,168 val / 3,005 test
Hospital_B: 9,074 train / 1,073 val / 508 test
Hospital_C: 29,813 train / 3,412 val / 1,711 test
Hospital_D: 1,276 train / 145 val / 74 test
Hospital_E: 2,997 train / 348 val / 168 test
✓ Image paths resolve correctly in this session


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


## Section 4 — GPU Optimisation

In [ ]:
torch.backends.cudnn.benchmark     = True
torch.backends.cudnn.deterministic = False

BATCH_SIZE  = 512
NUM_WORKERS = 4
PREFETCH    = 2

print(f"✓ BATCH_SIZE  : {BATCH_SIZE}")
print(f"✓ NUM_WORKERS : {NUM_WORKERS}")
print(f"✓ PREFETCH    : {PREFETCH}")

✓ BATCH_SIZE  : 512
✓ NUM_WORKERS : 4
✓ PREFETCH    : 2


## Section 5 — Transforms, Dataset

In [ ]:
IMAGE_SIZE = 224

train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.Grayscale(num_output_channels=3),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std =[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std =[0.229, 0.224, 0.225])
])

print(f"✓ Image size    : {IMAGE_SIZE}×{IMAGE_SIZE}")
print(f"✓ Normalisation : ImageNet mean/std")

class ChestXrayDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df        = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row   = self.df.iloc[idx]
        image = Image.open(row['image_path']).convert('RGB')
        if self.transform:
            image = self.transform(image)
        label = torch.tensor(row['label'], dtype=torch.float32)
        return image, label

print("✓ ChestXrayDataset defined")

✓ Image size    : 224×224
✓ Normalisation : ImageNet mean/std
✓ ChestXrayDataset defined


## Section 6 —  Model

In [ ]:
ROUNDS = 10

def build_model():
    model    = models.resnet18(weights='IMAGENET1K_V1')
    model.fc = nn.Linear(model.fc.in_features, 1)
    return model

test_model = build_model()
params     = sum(p.numel() for p in test_model.parameters())
print(f"✓ ResNet-18 — ImageNet pretrained")
print(f"✓ Parameters  : {params:,}")
print(f"✓ Rounds      : {ROUNDS}")
del test_model

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 239MB/s]


✓ ResNet-18 — ImageNet pretrained
✓ Parameters  : 11,177,025
✓ Rounds      : 10


## Section 7 — Evaluation Function

In [ ]:
def evaluate_client(model, dataframe, device):
    model = model.to(device)
    model.eval()

    loader = DataLoader(
        ChestXrayDataset(dataframe, transform=val_transform),
        batch_size         = BATCH_SIZE,
        shuffle            = False,
        num_workers        = NUM_WORKERS,
        pin_memory         = True,
        persistent_workers = True
    )

    all_probs, all_labels = [], []
    with torch.no_grad():
        for images, labels in loader:
            probs = torch.sigmoid(
                model(images.to(device, non_blocking=True))
            ).cpu().numpy()
            all_probs.extend(probs.flatten())
            all_labels.extend(labels.numpy())

    probs  = np.array(all_probs)
    labels = np.array(all_labels)
    preds  = (probs > 0.5).astype(int)

    def auc_fnr_for_mask(y_true, y_prob, y_pred):
        if len(y_true) < 10 or y_true.sum() == 0:
            return float('nan'), float('nan')
        try:
            auc = float(roc_auc_score(y_true, y_prob))
        except Exception:
            auc = float('nan')
        fn  = int(((y_pred == 0) & (y_true == 1)).sum())
        tp  = int(((y_pred == 1) & (y_true == 1)).sum())
        fnr = fn / (fn + tp) if (fn + tp) > 0 else 0.0
        return round(auc, 4), round(fnr, 4)

    auc, fnr = auc_fnr_for_mask(labels, probs, preds)
    acc      = round(float((preds == labels).mean() * 100), 2)
    metrics  = {'auc': auc, 'fnr': fnr, 'accuracy': acc}

    for sex in ['M', 'F']:
        mask = dataframe['Patient Sex'].values == sex
        if mask.sum() > 10:
            a, f = auc_fnr_for_mask(labels[mask], probs[mask], preds[mask])
            metrics[f'auc_{sex}'] = a
            metrics[f'fnr_{sex}'] = f

    for grp in ['0-20', '20-40', '40-60', '60-80', '80+']:
        mask = dataframe['Age Group'].values == grp
        if mask.sum() > 10:
            a, f = auc_fnr_for_mask(labels[mask], probs[mask], preds[mask])
            metrics[f'auc_{grp}'] = a
            metrics[f'fnr_{grp}'] = f

    return metrics

print("✓ evaluate_client() defined")
print("  Metrics : AUC + FNR (overall, per sex, per age group)")

✓ evaluate_client() defined
  Metrics : AUC + FNR (overall, per sex, per age group)


## Section 8 — Flower Base Client

In [ ]:
class HospitalClient(fl.client.NumPyClient):

    def __init__(self, name: str, dataframe, val_dataframe, device):
        self.name          = name
        self.dataframe     = dataframe      # training data
        self.val_dataframe = val_dataframe  # validation data
        self.device        = device
        self.model         = build_model().to(device)

    def get_parameters(self, config) -> NDArrays:
        return [v.cpu().numpy()
                for v in self.model.state_dict().values()]

    def set_parameters(self, parameters: NDArrays):
        state_dict = dict(zip(
            self.model.state_dict().keys(),
            [torch.tensor(p) for p in parameters]
        ))
        self.model.load_state_dict(state_dict, strict=True)

    def fit(self, parameters: NDArrays, config: Dict) -> Tuple[NDArrays, int, Dict]:
        self.set_parameters(parameters)
        self.model.train()

        epochs = int(config.get('epochs', 3))
        lr     = float(config.get('lr', 1e-4))

        loader = DataLoader(
            ChestXrayDataset(self.dataframe, transform=train_transform),
            batch_size         = BATCH_SIZE,
            shuffle            = True,
            num_workers        = NUM_WORKERS,
            pin_memory         = True,
            persistent_workers = True,
            prefetch_factor    = PREFETCH
        )

        criterion = nn.BCEWithLogitsLoss()
        optimiser = torch.optim.Adam(self.model.parameters(), lr=lr)

        total_loss, total_samples = 0.0, 0
        for _ in range(epochs):
            for images, labels in loader:
                images  = images.to(self.device, non_blocking=True)
                labels  = labels.to(self.device, non_blocking=True).unsqueeze(1)
                outputs = self.model(images)
                loss    = criterion(outputs, labels)
                optimiser.zero_grad()
                loss.backward()
                optimiser.step()
                total_loss    += loss.item() * len(labels)
                total_samples += len(labels)

        avg_loss = total_loss / total_samples
        return (
            self.get_parameters(config={}),
            len(self.dataframe),
            {'loss': float(avg_loss), 'client_name': self.name}
        )

    def evaluate(self, parameters: NDArrays, config: Dict) -> Tuple[float, int, Dict]:
        self.set_parameters(parameters)
        m = evaluate_client(self.model, self.val_dataframe, self.device)
        m['client_name'] = self.name
        return float(1.0 - (m['auc'] if not np.isnan(m['auc']) else 0.5)), \
                len(self.val_dataframe), m

print("✓ HospitalClient defined")
print("  fit()      → trains on train_clients")
print("  evaluate() → validates on val_clients after each round")

✓ HospitalClient defined
  fit()      → trains on train_clients
  evaluate() → validates on val_clients after each round


## Section 9 — FedAvg (McMahan et al. 2017)
Algorithm 1: Server aggregates w_{t+1} = Σ_k (n_k/n) · w_k^{t+1}

In [ ]:
def make_client_fn(train_data_map, val_data_map, device):
    def client_fn(cid: str) -> fl.client.Client:
        name = HOSPITAL_NAMES[int(cid)]
        return HospitalClient(
            name          = name,
            dataframe     = train_data_map[name],
            val_dataframe = val_data_map[name],
            device        = device
        ).to_client()
    return client_fn

print(f"✓ client_fn factory defined")

✓ client_fn factory defined


In [ ]:
import logging
logging.getLogger('flwr').setLevel(logging.ERROR)
import warnings
warnings.filterwarnings('ignore')

class FedAvgWithSave(fl.server.strategy.FedAvg):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.final_parameters = None

    def aggregate_fit(self, server_round, results, failures):
        aggregated = super().aggregate_fit(server_round, results, failures)
        if aggregated[0] is not None:
            self.final_parameters = aggregated[0]
        return aggregated

    def aggregate_evaluate(self, server_round, results, failures):
        aggregated = super().aggregate_evaluate(server_round, results, failures)

        if results:
            print(f"\n── Round {server_round}/{ROUNDS} Validation ──")
            for _, res in results:
                name = res.metrics.get('client_name', '?')
                auc  = res.metrics.get('auc', float('nan'))
                fnr  = res.metrics.get('fnr', float('nan'))
                print(f"  {name:<14} AUC: {auc:.4f}  FNR: {fnr:.4f}")
            aucs       = [r.metrics.get('auc', float('nan')) for _, r in results]
            valid_aucs = [a for a in aucs if not (a != a)]
            if valid_aucs:
                print(f"  Mean AUC : {sum(valid_aucs)/len(valid_aucs):.4f} | "
                      f"Variance : {float(np.var(valid_aucs)):.6f}")

        return aggregated

fedavg_strategy = FedAvgWithSave(
    fraction_fit          = 1.0,
    fraction_evaluate     = 1.0,
    min_fit_clients       = NUM_CLIENTS,
    min_evaluate_clients  = NUM_CLIENTS,
    min_available_clients = NUM_CLIENTS,
    on_fit_config_fn      = lambda rnd: {'epochs': 3, 'lr': 1e-4}
)
print("✓ FedAvg strategy ready [McMahan et al. 2017]")
print(f"  Rounds: {ROUNDS} | Epochs/round: 3 | Clients: {NUM_CLIENTS}")
print(f"  Per-round validation output enabled")

✓ FedAvg strategy ready [McMahan et al. 2017]
  Rounds: 10 | Epochs/round: 3 | Clients: 5
  Per-round validation output enabled


In [ ]:
import os, logging
os.environ['RAY_SILENT_MODE'] = '1'
logging.getLogger('flwr').setLevel(logging.ERROR)

print("=" * 50)
print("FedAvg — McMahan et al. 2017")
print(f"Rounds: {ROUNDS} | Epochs/round: 3 | Clients: {NUM_CLIENTS}")
print(f"Split: 85/10/5 | Batch: {BATCH_SIZE}")
print("=" * 50)

t0 = time.time()

fedavg_history = fl.simulation.start_simulation(
    client_fn        = make_client_fn(train_clients, val_clients, device),
    num_clients      = NUM_CLIENTS,
    config           = fl.server.ServerConfig(num_rounds=ROUNDS),
    strategy         = fedavg_strategy,
    client_resources = {'num_gpus': 1.0}
)

elapsed = time.time() - t0
print(f"\n{'='*50}")
print(f"✓ FedAvg complete in {elapsed/60:.1f} minutes")
print(f"\nLoss per round:")
for rnd, loss in fedavg_history.losses_distributed:
    print(f"  Round {rnd:>2} : {loss:.4f}")

FedAvg — McMahan et al. 2017
Rounds: 10 | Epochs/round: 3 | Clients: 5
Split: 85/10/5 | Batch: 512


2026-07-18 12:55:46,088	INFO worker.py:2012 -- Started a local Ray instance.
(ClientAppActor pid=16053) WARNING :   DEPRECATED FEATURE: `client_fn` now expects a signature `def client_fn(context: Context)`.The provided `client_fn` has signature: {'cid': <Parameter "cid: str">}. You can import the `Context` like this: `from flwr.app import Context`
(ClientAppActor pid=16053) 
(ClientAppActor pid=16053)             This is a deprecated feature. It will be removed
(ClientAppActor pid=16053)             entirely in future versions of Flower.
(ClientAppActor pid=16053)         
(ClientAppActor pid=16053) WARNING :   DEPRECATED FEATURE: `client_fn` now expects a signature `def client_fn(context: Context)`.The provided `client_fn` has signature: {'cid': <Parameter "cid: str">}. You can import the `Context` like this: `from flwr.app import Context`
(ClientAppActor pid=16053) 
(ClientAppActor pid=16053)             This is a deprecated feature. It will be removed
(ClientAppActor pid=16053)     


── Round 1/10 Validation ──
  Hospital_B     AUC: 0.8439  FNR: 0.3626
  Hospital_D     AUC: 0.7085  FNR: 0.4904
  Hospital_E     AUC: 0.6689  FNR: 0.4750
  Hospital_C     AUC: 0.7732  FNR: 0.3297
  Hospital_A     AUC: 0.7421  FNR: 0.3705
  Mean AUC : 0.7473 | Variance : 0.003536


(ClientAppActor pid=16053) WARNING :   DEPRECATED FEATURE: `client_fn` now expects a signature `def client_fn(context: Context)`.The provided `client_fn` has signature: {'cid': <Parameter "cid: str">}. You can import the `Context` like this: `from flwr.app import Context`
(ClientAppActor pid=16053) 
(ClientAppActor pid=16053)             This is a deprecated feature. It will be removed
(ClientAppActor pid=16053)             entirely in future versions of Flower.
(ClientAppActor pid=16053)         
(ClientAppActor pid=16053) /usr/lib/python3.12/multiprocessing/popen_fork.py:66: DeprecationWarning: This process (pid=16053) is multi-threaded, use of fork() may lead to deadlocks in the child.
(ClientAppActor pid=16053)   self.pid = os.fork()
(ClientAppActor pid=16053) WARNING :   DEPRECATED FEATURE: `client_fn` now expects a signature `def client_fn(context: Context)`.The provided `client_fn` has signature: {'cid': <Parameter "cid: str">}. You can import the `Context` like this: `from flwr


── Round 2/10 Validation ──
  Hospital_B     AUC: 0.8636  FNR: 0.4187
  Hospital_D     AUC: 0.7298  FNR: 0.5096
  Hospital_A     AUC: 0.7487  FNR: 0.4252
  Hospital_E     AUC: 0.6808  FNR: 0.4750
  Hospital_C     AUC: 0.7734  FNR: 0.3978
  Mean AUC : 0.7593 | Variance : 0.003644


...


── Round 8/10 Validation ──
  Hospital_B     AUC: 0.7891  FNR: 0.4112
  Hospital_A     AUC: 0.7280  FNR: 0.4074
  Hospital_D     AUC: 0.7312  FNR: 0.4423
  Hospital_C     AUC: 0.7370  FNR: 0.4014
  Hospital_E     AUC: 0.7086  FNR: 0.4500
  Mean AUC : 0.7388 | Variance : 0.000724


(ClientAppActor pid=16053) WARNING :   DEPRECATED FEATURE: `client_fn` now expects a signature `def client_fn(context: Context)`.The provided `client_fn` has signature: {'cid': <Parameter "cid: str">}. You can import the `Context` like this: `from flwr.app import Context`
(ClientAppActor pid=16053) 
(ClientAppActor pid=16053)             This is a deprecated feature. It will be removed
(ClientAppActor pid=16053)             entirely in future versions of Flower.
(ClientAppActor pid=16053)         
(ClientAppActor pid=16053) /usr/lib/python3.12/multiprocessing/popen_fork.py:66: DeprecationWarning: This process (pid=16053) is multi-threaded, use of fork() may lead to deadlocks in the child.
(ClientAppActor pid=16053)   self.pid = os.fork()
(ClientAppActor pid=16053) WARNING :   DEPRECATED FEATURE: `client_fn` now expects a signature `def client_fn(context: Context)`.The provided `client_fn` has signature: {'cid': <Parameter "cid: str">}. You can import the `Context` like this: `from flwr


── Round 9/10 Validation ──
  Hospital_C     AUC: 0.7147  FNR: 0.4839
  Hospital_B     AUC: 0.8321  FNR: 0.4916
  Hospital_E     AUC: 0.7385  FNR: 0.4250
  Hospital_A     AUC: 0.7147  FNR: 0.4879
  Hospital_D     AUC: 0.7003  FNR: 0.5385
  Mean AUC : 0.7401 | Variance : 0.002268


(ClientAppActor pid=16053) WARNING :   DEPRECATED FEATURE: `client_fn` now expects a signature `def client_fn(context: Context)`.The provided `client_fn` has signature: {'cid': <Parameter "cid: str">}. You can import the `Context` like this: `from flwr.app import Context`
(ClientAppActor pid=16053) 
(ClientAppActor pid=16053)             This is a deprecated feature. It will be removed
(ClientAppActor pid=16053)             entirely in future versions of Flower.
(ClientAppActor pid=16053)         
(ClientAppActor pid=16053) /usr/lib/python3.12/multiprocessing/popen_fork.py:66: DeprecationWarning: This process (pid=16053) is multi-threaded, use of fork() may lead to deadlocks in the child.
(ClientAppActor pid=16053)   self.pid = os.fork()
(ClientAppActor pid=16053) WARNING :   DEPRECATED FEATURE: `client_fn` now expects a signature `def client_fn(context: Context)`.The provided `client_fn` has signature: {'cid': <Parameter "cid: str">}. You can import the `Context` like this: `from flwr


── Round 10/10 Validation ──
  Hospital_A     AUC: 0.7168  FNR: 0.4359
  Hospital_D     AUC: 0.7228  FNR: 0.4519
  Hospital_E     AUC: 0.7194  FNR: 0.4000
  Hospital_C     AUC: 0.7125  FNR: 0.4552
  Hospital_B     AUC: 0.8237  FNR: 0.4290
  Mean AUC : 0.7390 | Variance : 0.001803

✓ FedAvg complete in 304.9 minutes

Loss per round:
  Round  1 : 0.2413
  Round  2 : 0.2350
  Round  3 : 0.2347
  Round  4 : 0.2375
  Round  5 : 0.2479
  Round  6 : 0.2426
  Round  7 : 0.2615
  Round  8 : 0.2639
  Round  9 : 0.2734
  Round 10 : 0.2741


In [ ]:
import gc, ray
if ray.is_initialized():
    ray.shutdown()
torch.cuda.empty_cache()
gc.collect()

fedavg_metrics = {}
final_params   = parameters_to_ndarrays(fedavg_strategy.final_parameters)

for name, data in test_clients.items():
    model = build_model().to(device)
    model.load_state_dict(dict(zip(
        model.state_dict().keys(),
        [torch.tensor(p) for p in final_params]
    )))
    fedavg_metrics[name] = evaluate_client(model, data, device)
    del model
    torch.cuda.empty_cache()

rows = []
for name in HOSPITAL_NAMES:
    m = fedavg_metrics[name]
    rows.append({
        'Client'   : name,
        'AUC'      : m['auc'],
        'FNR'      : m['fnr'],
        'Accuracy' : m['accuracy'],
        'AUC_M'    : m.get('auc_M', 'N/A'),
        'FNR_M'    : m.get('fnr_M', 'N/A'),
        'AUC_F'    : m.get('auc_F', 'N/A'),
        'FNR_F'    : m.get('fnr_F', 'N/A'),
    })

df_results = pd.DataFrame(rows).set_index('Client')
print("=== FedAvg Results — McMahan et al. 2017 ===\n")
print(df_results.to_string())

valid_aucs = [fedavg_metrics[n]['auc'] for n in HOSPITAL_NAMES
              if not (fedavg_metrics[n]['auc'] != fedavg_metrics[n]['auc'])]
auc_var = np.var(valid_aucs) if valid_aucs else float('nan')
print(f"\nAUC Variance (valid hospitals only) : {auc_var:.6f}")
print(f"Valid hospitals : {len(valid_aucs)}/5")

=== FedAvg Results — McMahan et al. 2017 ===

               AUC     FNR  Accuracy   AUC_M   FNR_M   AUC_F   FNR_F
Client                                                              
Hospital_A  0.7553  0.6676     57.04  0.7688  0.6534  0.7366  0.6872
Hospital_B     NaN  0.6811     31.89     NaN  0.6593     NaN  0.7064
Hospital_C  0.7427  0.7259     85.91  0.7816  0.7000  0.6978  0.7538
Hospital_D  0.6652  0.6923     48.65  0.8061  0.6970  0.5789  0.6842
Hospital_E  0.7778  0.6316     89.88  0.8067  0.6000  0.7514  0.6667

AUC Variance (valid hospitals only) : 0.001794
Valid hospitals : 4/5


In [ ]:
SAVE_DIR = '/content/drive/MyDrive/dissertation/results'
os.makedirs(SAVE_DIR, exist_ok=True)

with open(f'{SAVE_DIR}/fedavg_full_metrics.json', 'w') as f:
    json.dump(fedavg_metrics, f, indent=2, default=str)

torch.save(
    dict(zip(build_model().state_dict().keys(),
             [torch.tensor(v) for v in final_params])),
    f'{SAVE_DIR}/fedavg_full_model.pth'
)

print("✓ FedAvg metrics saved → fedavg_full_metrics.json")
print("✓ FedAvg model saved  → fedavg_full_model.pth")

✓ FedAvg metrics saved → fedavg_full_metrics.json
✓ FedAvg model saved  → fedavg_full_model.pth
